# 01 — Exploring the Data Lake

A reusable notebook for answering *"what data is here and what's in it?"* — for
Solar Analytics now, and any future source (SAPN, BOM, etc.).

**The three zoom levels:**

| Level | Question | Tool | Cost |
|---|---|---|---|
| **Storage** | What files physically exist? | `s3_ls()` (boto3) | free |
| **Catalog** | What's registered as a queryable table? | `databases()`, `tables()`, `describe()` (Glue) | free |
| **Data** | What do the values look like? | `aq()` (Athena) or `dread()` (DuckDB) | small / local |

Reminder: if anything says *"token expired"*, run `aws sso login --profile ciccada`
in a terminal and re-run the cell.

In [ ]:
from aws_config import *
session.client('sts').get_caller_identity()['Arn']  # confirm you're logged in

---
## Level 1 — Storage: the physical files in S3

`s3_ls()` lists what's actually in the bucket. This is ground truth: it works
even for data that isn't in Glue yet, so it's where you start with any *new*
source. Notice that "folders" are just naming — each table is a folder of
Parquet files.

In [ ]:
# Top level of the bucket — the major data areas
s3_ls()

In [ ]:
# Inside the Spark warehouse — where results tables are written
s3_ls('spark-warehouse/')

In [ ]:
# Drill into one table's folder to SEE its partitioning.
# Partition folders look like year=2025/month=1/ — this is what makes
# `WHERE year=2025 AND month=1` cheap (Athena reads only those folders).
s3_ls('Trino-Warehouse/')

---
## Level 2 — Catalog: what's queryable, and its schema

Glue is the index that lets you write SQL. These calls are free (metadata only).

In [ ]:
databases()

In [ ]:
# Every table in every database, in one view.
import pandas as pd
all_tabs = pd.concat([tables(d) for d in databases()['Database']], ignore_index=True)
all_tabs[['Database', 'Table']]

In [ ]:
# The columns + types of any table. Works for Iceberg tables too (where the
# Glue column list is often blank). This is your 'data dictionary' lookup.
describe('ts', database='solar_analytics_iceberg')

---
## Level 3 — Data: peek at actual rows

Two engines, same files. Use `aq()` (Athena) for normal SQL; use `dread()`
(DuckDB, reads the Parquet file directly) for quick peeks or when Glue's
description is wrong.

**Cost discipline lives here:** on the big `ts` table, always filter on
`year`/`month`/`is_pv` and never sort on a fresh peek.

In [ ]:
# Dimension tables are tiny — peek freely
aq('SELECT * FROM circuits LIMIT 5')

In [ ]:
# The telemetry fact table: a CHEAP peek (no ORDER BY, just LIMIT).
# This shows the real column names so we can write proper filtered queries next.
aq('SELECT * FROM ts LIMIT 5', database='solar_analytics_iceberg')

In [ ]:
# How many rows in one month of PV telemetry? (count scans only the partition)
aq('''
    SELECT count(*) AS rows
    FROM ts
    WHERE is_pv = True AND year = 2025 AND month = 1
''', database='solar_analytics_iceberg')

---
## When Athena chokes: read the Parquet directly with DuckDB

Some results tables have *schema drift* — the Glue description disagrees with
the file (e.g. a column the file stores as integer but Glue calls a double).
Athena refuses; DuckDB reads the file's own schema and just works.

The S3 path comes from the error message, or from `s3_ls()`.

In [ ]:
dread('s3://project-ciccada/spark-warehouse/'
      'Compliance_results_SolA/compliance_voltvar.parquet/*.parquet')

---
## Recipe: exploring ANY new source in future

1. **`s3_ls('<prefix>/')`** — see what physically exists and how it's foldered.
2. **`databases()` / `tables(db)`** — is it registered in Glue? If yes, you can use SQL.
3. **`describe('<table>', db)`** — learn its columns.
4. **`aq('SELECT ... LIMIT 5')`** — peek at values (filter on partitions if it's big).
5. **Not in Glue, or Glue is wrong?** — point **`dread('s3://.../*.parquet')`** straight at the files.